<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/exams/midterm/MID3_evolving_vehicles.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Midterm Capstone: Machines, Tests and Meaning

## Project 3: Evolving Braitenberg vehicles

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

Instead of wiring vehicles by hand, a genetic algorithm evolves sensor-motor weights and the angle of the sensors for two tasks: approaching a light and keeping a fixed distance from it. The evolved controllers are then analysed in Braitenberg's terms, which illustrates the difference between downhill invention and uphill analysis [1, 2].

**Required extensions.** Add a second light with a different colour and sensors for each colour, evolve controllers that approach one light while avoiding the other, and study how the sensor angle, a property of the body, co-evolves with the wiring. Discuss whether observers' descriptions in terms of fear or love are justified and what the results imply for embodied accounts of cognition [3, 4].

## What this example implements

The notebook simulates whole populations of vehicles at once with NumPy, evolves them with tournament selection, elitism and Gaussian mutation for two tasks, describes the dominant wiring of the best controllers and plots their trajectories.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Evolution of controllers

In [ ]:
rng = np.random.default_rng(SEED)
LIGHTS = [(4.0, 3.0)]
STARTS = np.array([[-6, -4, 0.3], [6, -6, 2.5], [-7, 6, -1.0], [0, -8, 1.6]])

def simulate(genomes, lights=LIGHTS, starts=STARTS, steps=300, dt=0.05, keep_path=False):
    P, S = len(genomes), len(starts)
    g, st = np.repeat(genomes, S, axis=0), np.tile(starts, (P, 1))
    x, y, th = st[:, 0].astype(float), st[:, 1].astype(float), st[:, 2].astype(float)
    phi = np.clip(g[:, 6], 0.05, 1.5)
    dist_sum, path = np.zeros(P * S), []
    for _ in range(steps):
        il, ir = [sum(1 / (1 + ((x + 0.5 * np.cos(th + s * phi) - lx) ** 2 + (y + 0.5 * np.sin(th + s * phi) - ly) ** 2) / 4) for lx, ly in lights) for s in (1, -1)]
        vl = 2 + 2 * np.tanh(g[:, 4] + g[:, 0] * il + g[:, 2] * ir)
        vr = 2 + 2 * np.tanh(g[:, 5] + g[:, 1] * il + g[:, 3] * ir)
        th = th + (vr - vl) * dt
        x = np.clip(x + (vl + vr) / 2 * np.cos(th) * dt, -10, 10)
        y = np.clip(y + (vl + vr) / 2 * np.sin(th) * dt, -10, 10)
        d = np.min([np.hypot(x - lx, y - ly) for lx, ly in lights], axis=0)
        dist_sum += d
        if keep_path:
            path.append((x.copy(), y.copy()))
    return (dist_sum / steps).reshape(P, S), path

def evolve(fitness, pop=60, gens=40, sigma=0.3):
    G = rng.normal(0, 1, (pop, 7)); G[:, 6] = rng.uniform(0.1, 1.2, pop)
    best = []
    for _ in range(gens):
        f = fitness(simulate(G)[0])
        order = np.argsort(-f); best.append(f[order[0]])
        elite = G[order[:4]]
        parents = G[[max(rng.choice(pop, 3), key=lambda i: f[i]) for _ in range(pop - 4)]]
        G = np.r_[elite, parents + rng.normal(0, sigma, parents.shape)]
    return G[0], best

approach, curve_a = evolve(lambda d: -d.mean(1))
orbit, curve_o = evolve(lambda d: -np.abs(d - 3).mean(1))
print("fitness at the last generation: approach", round(curve_a[-1], 3), "| keep a distance of 3:", round(curve_o[-1], 3))

## 2. Analysing what evolved

In [ ]:
def describe(g):
    uncrossed, crossed = g[0] + g[3], g[1] + g[2]
    kind = "crossed" if abs(crossed) > abs(uncrossed) else "uncrossed"
    sign = "excitatory" if (crossed if kind == "crossed" else uncrossed) > 0 else "inhibitory"
    return f"dominant connections {kind} and {sign}; weights ll {g[0]:.2f}, lr {g[1]:.2f}, rl {g[2]:.2f}, rr {g[3]:.2f}; sensor angle {g[6]:.2f} rad"

print("approach task:", describe(approach))
print("distance task:", describe(orbit))
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
for a, g, title in [(ax[0], approach, "evolved to approach"), (ax[1], orbit, "evolved to keep a distance")]:
    _, path = simulate(g[None, :], keep_path=True)
    xs, ys = np.array([p[0] for p in path]), np.array([p[1] for p in path])
    for s in range(len(STARTS)):
        a.plot(xs[:, s], ys[:, s])
    a.plot(*LIGHTS[0], "*", ms=16, color="orange"); a.set_xlim(-10, 10); a.set_ylim(-10, 10); a.set_aspect("equal"); a.set_title(title)
plt.show()

## Report and submission

Both tracks follow the course report template. A research report states its thesis in the introduction, reconstructs the relevant arguments as explicit premises and conclusions, and evaluates them. An application report explains what the simulation or experiment can and cannot show about the philosophical question, presents its results with figures and discusses its limitations. All sources are cited in square brackets.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the midterm capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 7, with the report and all code files attached or linked.

## References

[1] Braitenberg, V. (1984). *Vehicles: Experiments in Synthetic Psychology*. MIT Press.

[2] Pfeifer, R., & Bongard, J. (2006). *How the Body Shapes the Way We Think: A New View of Intelligence*. MIT Press.

[3] Brooks, R. A. (1991). Intelligence without representation. *Artificial Intelligence*, 47(1-3), 139-159. <https://doi.org/10.1016/0004-3702(91)90053-M>

[4] Varela, F. J., Thompson, E., & Rosch, E. (1991). *The Embodied Mind: Cognitive Science and Human Experience*. MIT Press.